# Lesson 7: Overfitting, Train/Test Split & Cross-Validation

This is the capstone of the question you've been asking since Lesson 4: **what does it really mean for a model to be 'right'?**

The answer: **generalization** — performing well on data it has NEVER seen.

We'll cover:
1. **Overfitting** vs **underfitting** (memorizing vs learning).
2. **Train/test split** — how we honestly measure generalization.
3. **Cross-validation** — a more reliable version.
4. **Regularization** — a tool to fight overfitting (callback to the differentiators list!).

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 1: The core problem — memorizing vs learning

- **Underfitting:** the model is too simple; it misses the real pattern. Bad on training AND new data.
- **Good fit:** captures the true pattern. Good on both.
- **Overfitting:** the model memorizes the training data (including its noise). Great on training, BAD on new data.

```
Underfit          Good fit           Overfit
  /                 .-'                 /\  /\
 /        vs      /        vs         /  \/  \  (wiggles through every noisy point)
```

A student analogy: overfitting = memorizing past exam answers word-for-word, then failing when the questions change. Learning the concepts = generalization.

## Step 2: Train/Test Split — the honest test

We HIDE part of the data from the model during training, then test on it. If the model does well on data it never saw, it truly generalized.

Typical split: 80% train, 20% test.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Make data with a clear linear trend + a little noise
rng = np.random.RandomState(42)
X = np.linspace(0, 10, 60).reshape(-1, 1)
y = 3 * X.ravel() + 7 + rng.normal(0, 3, size=60)

# Hide 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples: {len(X_train)},  Test samples: {len(X_test)}')

model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate on BOTH sets
train_score = r2_score(y_train, model.predict(X_train))
test_score = r2_score(y_test, model.predict(X_test))
print(f'R2 on training data: {train_score:.3f}')
print(f'R2 on TEST data:     {test_score:.3f}   <- the number that actually matters')

**R2 (R-squared)** ranges up to 1.0 (perfect). The **test score** is the honest measure of quality. If train score is high but test score is low -> overfitting.

## Step 3: See overfitting happen (high-degree polynomial)

We'll fit polynomials of increasing complexity. A too-complex model wiggles through every noisy point: tiny training error but big TEST error.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

print(f"{'degree':>7} {'train_MSE':>12} {'test_MSE':>12}")
for degree in [1, 2, 5, 10, 15]:
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    poly_model.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, poly_model.predict(X_train))
    test_mse = mean_squared_error(y_test, poly_model.predict(X_test))
    print(f'{degree:>7} {train_mse:>12.2f} {test_mse:>12.2f}')

print('\nWatch: as degree grows, train_MSE keeps dropping but test_MSE eventually EXPLODES = overfitting.')

## Step 4: Cross-Validation — a more reliable estimate

A single train/test split can be lucky or unlucky. **K-fold cross-validation** splits the data into K parts, trains K times (each time using a different part as the test set), and averages the scores. More trustworthy.

```
5-fold:  [test ][train][train][train][train]
         [train][test ][train][train][train]
         ... rotate the test fold 5 times, average the scores ...
```

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print('R2 for each of the 5 folds:', np.round(scores, 3))
print(f'Average R2: {scores.mean():.3f}  (+/- {scores.std():.3f})')

## Step 5: Regularization — fighting overfitting

(Callback to the 'differentiators' list!) Regularization adds a penalty for large weights, discouraging the model from over-complicating itself.

- **Ridge (L2):** penalizes the sum of squared weights -> shrinks weights toward 0.
- **Lasso (L1):** penalizes the sum of absolute weights -> can push some weights exactly to 0 (automatic feature selection!).

The strength is a **hyperparameter** (`alpha`): bigger alpha = more penalty = simpler model.

In [ ]:
from sklearn.linear_model import Ridge

# Use the overfit-prone degree-15 polynomial, but add Ridge regularization
for alpha in [0.0, 0.001, 1.0]:
    reg = make_pipeline(PolynomialFeatures(15), Ridge(alpha=alpha))
    reg.fit(X_train, y_train)
    test_mse = mean_squared_error(y_test, reg.predict(X_test))
    label = '(no regularization)' if alpha == 0 else ''
    print(f'alpha={alpha:<6} -> test_MSE={test_mse:>10.2f} {label}')

print('\nMore regularization tames the wild degree-15 model and improves the TEST error.')

## Your turn

1. In Step 3, which polynomial degree gives the best TEST MSE? That's the 'sweet spot' between under- and overfitting.
2. Change `test_size` to 0.5 in Step 2. Do the scores change? Why might too-small a training set hurt?
3. In Step 5, what happens to test error if you set alpha very high (e.g. 1000)? (Hint: too much regularization -> underfitting.)
4. In your own words: why is the TEST score the one that matters, not the training score?

## The professional workflow: split + cross-validate to tune + final test

This is how the three pieces combine in real projects. The test set stays untouched until the very end; cross-validation picks the hyperparameter (`alpha`).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge

# Use the degree-5 polynomial features so regularization has something to tame
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# 1. Hold out a final test set (already have X_train/X_test from Step 2)

# 2. Cross-validate on the TRAINING set to pick the best alpha
print("Tuning alpha with 5-fold cross-validation:")
best_alpha, best_cv = None, -1e9
for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    pipe = make_pipeline(PolynomialFeatures(5), Ridge(alpha=alpha))
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2').mean()
    print(f'  alpha={alpha:<7} CV R2={cv:.3f}')
    if cv > best_cv:
        best_cv, best_alpha = cv, alpha

print(f'\nBest alpha = {best_alpha}')

# 3. Train final model with best alpha, evaluate ONCE on the untouched test set
final = make_pipeline(PolynomialFeatures(5), Ridge(alpha=best_alpha))
final.fit(X_train, y_train)
print(f'Final TEST R2 = {final.score(X_test, y_test):.3f}')